# Combined Deepfake Detection: Attention + Feature Selection (Trained)


---

### Cell 2: Code (Check Data & Setup)

In [1]:
# FIX: Add this line to resolve the AttributeError
!pip install protobuf==3.20.0

!ls /kaggle/input/1000-videos-split/1000_videos/train/fake | head -n 5
!ls /kaggle/input/1000-videos-split/1000_videos/train/real | head -n 5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 2.3 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 3.20.0 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.0 which is incompatible.
google-cloud-secret-manager 2.25.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 3.20.0 which is incompatible.
goog

### Hardware Detection

### Cell 2: Code (Check Data & Setup)

This cell runs a check to identify the available hardware and selects the appropriate TensorFlow distribution strategy.

In [2]:
import os
import tensorflow as tf

# Define your data path based on your Kaggle dataset
DATA_PATH = "/kaggle/input/1000-videos-split/1000_videos"

# Check if the directories exist
print("--- Checking Data Paths ---")
print(f"Train Real exists: {os.path.exists(os.path.join(DATA_PATH, 'train', 'real'))}")
print(f"Train Fake exists: {os.path.exists(os.path.join(DATA_PATH, 'train', 'fake'))}")
print(f"Val Real exists:   {os.path.exists(os.path.join(DATA_PATH, 'validation', 'real'))}")
print(f"Val Fake exists:   {os.path.exists(os.path.join(DATA_PATH, 'validation', 'fake'))}")
print(f"Test Real exists:  {os.path.exists(os.path.join(DATA_PATH, 'test', 'real'))}")
print(f"Test Fake exists:  {os.path.exists(os.path.join(DATA_PATH, 'test', 'fake'))}")
print("---------------------------")

def get_distribution_strategy():
    """
    Detects available hardware (TPU, multi-GPU, single-GPU, CPU) and returns
    the appropriate TensorFlow distribution strategy.
    """
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
        strategy = tf.distribute.TPUStrategy(tpu)
        print("✅ Running on TPU")
    except (ValueError, tf.errors.NotFoundError):
        gpus = tf.config.list_physical_devices('GPU')
        if len(gpus) > 1:
            strategy = tf.distribute.MirroredStrategy()
            print(f"✅ Running on {len(gpus)} GPUs")
        elif len(gpus) == 1:
            strategy = tf.distribute.get_strategy()
            print("✅ Running on a single GPU")
        else:
            strategy = tf.distribute.get_strategy()
            print("✅ Running on CPU")
            
    print(f"Number of accelerator replicas: {strategy.num_replicas_in_sync}")
    return strategy
    
# Run the detection function to see what hardware is available
strategy = get_distribution_strategy()

2025-11-12 17:38:30.264502: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762969110.471536      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762969110.522486      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


--- Checking Data Paths ---
Train Real exists: True
Train Fake exists: True
Val Real exists:   True
Val Fake exists:   True
Test Real exists:  True
Test Fake exists:  True
---------------------------
✅ Running on a single GPU
Number of accelerator replicas: 1


## Part 1: Write Python Modules

These cells will write the project's logic into separate `.py` files.


---

### Cell 4: Code (Write `model.py`)
*This is now completely different. It builds **one** model with **three** inputs.*

In [3]:
%%writefile model.py
import tensorflow as tf
from tensorflow.keras.layers import (
    Dense,
    Conv2D,
    BatchNormalization,
    Dropout,
    Reshape,
    Add,
    Flatten,
    Input,
    Concatenate
)
from tensorflow.keras.models import Model
import tensorflow.keras.applications.densenet as densenet
import tensorflow.keras.applications.efficientnet as efficientnet
import tensorflow.keras.applications.xception as xception

# --- Custom Attention Layers (from 2022 paper) ---

class ModifiedBranch(tf.keras.layers.Layer):
    def __init__(self, a_vec_size, **kwargs):
        super(ModifiedBranch, self).__init__(**kwargs)
        self.a_vec_size = a_vec_size
    
    def build(self, input_shape):
        self.dense_layer = Dense(self.a_vec_size, activation="tanh", name="att_mod_dense")
        super(ModifiedBranch, self).build(input_shape)

    def call(self, input):
        af = tf.keras.backend.mean(input, axis=2)
        hs = self.dense_layer(af)
        return hs

    def get_config(self):
        config = super().get_config()
        config.update({"a_vec_size": self.a_vec_size})
        return config


class MainBranch(tf.keras.layers.Layer):
    def __init__(self, a_vec_size, dim, **kwargs):
        super(MainBranch, self).__init__(**kwargs)
        self.a_vec_size = a_vec_size
        self.dim = dim

    def build(self, input_shape):
        self.reshape1 = Reshape((-1, self.a_vec_size), name="att_main_reshape1")
        self.relu = tf.keras.activations.relu
        self.dropout = Dropout(0.5, name="att_main_dropout")
        self.reshape2 = Reshape((self.dim**2, self.a_vec_size), name="att_main_reshape2")
        super(MainBranch, self).build(input_shape)

    def call(self, input):
        e = tf.transpose(input, perm=[0, 2, 1])
        e = self.reshape1(e)
        e = self.relu(e)
        e = self.dropout(e)
        e = self.reshape2(e)
        e = tf.transpose(e, perm=[0, 2, 1])
        return e

    def get_config(self):
        config = super().get_config()
        config.update({"a_vec_size": self.a_vec_size, "dim": self.dim})
        return config


class Attention(tf.keras.layers.Layer):
    def __init__(self, dim, a_vec_size, **kwargs):
        super(Attention, self).__init__(**kwargs)
        self.dim = dim
        self.a_vec_size = a_vec_size
    
    def build(self, input_shape):
        self.dense1 = Dense(self.dim**2, name="att_att_dense1")
        self.reshape1 = Reshape((1, self.dim**2), name="att_att_reshape1")
        self.add = Add(name="att_att_add")
        self.dropout = Dropout(0.5, name="att_att_dropout")
        self.relu = tf.keras.activations.relu
        self.reshape2 = Reshape((-1, self.a_vec_size), name="att_att_reshape2")
        self.dense2 = Dense(1, use_bias=False, name="att_att_dense2")
        self.reshape3 = Reshape((-1, self.dim**2), name="att_att_reshape3")
        super(Attention, self).build(input_shape)

    def call(self, input):
        eh = self.dense1(input[0])
        eh = self.reshape1(eh)
        eh = self.add([input[1], eh])
        eh = self.relu(eh)
        eh = self.dropout(eh)
        eh = tf.transpose(eh, perm=[0, 2, 1])
        eh = self.reshape2(eh)
        eh = self.dense2(eh)
        eh = self.reshape3(eh)
        eh = self.relu(eh)
        return eh

    def get_config(self):
        config = super().get_config()
        config.update({"dim": self.dim, "a_vec_size": self.a_vec_size})
        return config

# --- Backbone Config ---
def get_backbone(backbone_name, input_shape=(299, 299, 3)):
    if backbone_name == "DenseNet121":
        base_model = densenet.DenseNet121(
            include_top=False, weights="imagenet", input_shape=input_shape
        )
        feature_map_layer = base_model.layers[-2].output
        dim = 9
        a_vec_size = 1024
        freeze_until = 311 # Freeze first ~75% of layers
    elif backbone_name == "EfficientNetB0":
        base_model = efficientnet.EfficientNetB0(
            include_top=False, weights="imagenet", input_shape=input_shape
        )
        feature_map_layer = base_model.layers[-3].output
        dim = 10
        a_vec_size = 1280
        freeze_until = 175 # Freeze first ~75% of layers
    elif backbone_name == "Xception":
        base_model = xception.Xception(
            include_top=False, weights="imagenet", input_shape=input_shape
        )
        feature_map_layer = base_model.layers[-13].output
        dim = 19
        a_vec_size = 1024
        freeze_until = 100 # Freeze first ~75% of layers
    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")

    # --- THIS IS THE FIX ---
    # 1. Set the entire model as trainable
    base_model.trainable = True
    
    # 2. Freeze all layers *except* the top ones
    for layer in base_model.layers[:freeze_until]:
        layer.trainable = False
        
    # 3. Re-freeze BatchNormalization layers (important for fine-tuning)
    for layer in base_model.layers:
        if isinstance(layer, BatchNormalization):
            layer.trainable = False
            
    return base_model, feature_map_layer, dim, a_vec_size

# --- Build Full Model ---

def build_attention_branch(base_model, feature_map_layer, dim, a_vec_size, name_prefix):
    """Creates the trainable attention branch for one backbone."""
    x = Conv2D(
        filters=a_vec_size,
        kernel_size=(1, 1),
        strides=(1, 1),
        padding="valid",
        use_bias=True,
        name=f"{name_prefix}_att_conv",
    )(feature_map_layer)
    x = BatchNormalization(axis=-1, name=f"{name_prefix}_att_bn")(x)
    x = tf.keras.activations.relu(x)
    x = Dropout(0.8, name=f"{name_prefix}_att_drop")(x)
    x = Reshape((a_vec_size, dim**2), name=f"{name_prefix}_att_reshape")(x)

    modified = ModifiedBranch(a_vec_size, name=f"{name_prefix}_mod_branch")(x)
    main = MainBranch(a_vec_size, dim, name=f"{name_prefix}_main_branch")(x)
    attention_features = Attention(dim, a_vec_size, name=f"{name_prefix}_attention")(
        [modified, main]
    )
    
    # --- THIS IS THE LINE I FIXED ---
    output_features = Flatten(name=f"{name_prefix}_flatten")(attention_features)
    
    # Return the full branch model
    return Model(inputs=base_model.input, outputs=output_features, name=f"{name_prefix}_branch")

def build_combined_model(input_shape=(299, 299, 3)):
    """
    Builds the final combined model with 3 inputs.
    """
    # Create the 3 inputs
    input_densenet = Input(shape=input_shape, name="densenet_input")
    input_efficientnet = Input(shape=input_shape, name="efficientnet_input")
    input_xception = Input(shape=input_shape, name="xception_input")
    
    # --- Branch 1: DenseNet121 ---
    base_d, feat_d, dim_d, avec_d = get_backbone("DenseNet121", input_shape)
    branch_d = build_attention_branch(base_d, feat_d, dim_d, avec_d, "densenet")
    
    # --- Branch 2: EfficientNetB0 ---
    base_e, feat_e, dim_e, avec_e = get_backbone("EfficientNetB0", input_shape)
    branch_e = build_attention_branch(base_e, feat_e, dim_e, avec_e, "efficientnet")
    
    # --- Branch 3: Xception ---
    base_x, feat_x, dim_x, avec_x = get_backbone("Xception", input_shape)
    branch_x = build_attention_branch(base_x, feat_x, dim_x, avec_x, "xception")
    
    # Get features from each branch
    features_d = branch_d(input_densenet)
    features_e = branch_e(input_efficientnet)
    features_x = branch_x(input_xception)
    
    # Combine the features
    combined_features = Concatenate(name="combined_features")([features_d, features_e, features_x])
    
    x = Dropout(0.5, name="pre_dense_dropout")(combined_features) 
    x = Dense(512, activation='relu', name="final_dense_1")(x)
    x = Dropout(0.5, name="final_dropout")(x)
    output = Dense(2, activation='softmax', name="final_output")(x)
    
    # Create the final model
    model = Model(
        inputs=[input_densenet, input_efficientnet, input_xception],
        outputs=output,
        name="Combined_Attention_Model"
    )
    
    return model

print("model.py written (v7 - Syntax Fix & Fine-Tuning).")

Writing model.py


**UTILITIES & FEATURES SELECTION**

In [4]:
%%writefile utils.py
import os
import glob
import numpy as np
import tensorflow as tf
from sklearn.utils import shuffle
import tensorflow.keras.applications.densenet as densenet
import tensorflow.keras.applications.efficientnet as efficientnet
import tensorflow.keras.applications.xception as xception
import tensorflow.keras.preprocessing.image as tf_image

# --- Data Loading ---

def prepare_dataset_paths(base_path, counts):
    """Prepare dataset paths and labels based on subfolders."""
    print(f"Loading data paths from: {base_path}")
    
    real_count = counts.get('real', 0)
    fake_count = counts.get('fake', 0)
    
    # Use 'validation' folder name for val split, as per your dataset
    split_name = 'validation' if 'val' in base_path else 'train'
    if 'test' in base_path: split_name = 'test'
        
    real_paths = sorted(glob.glob(os.path.join(base_path, "real", "*.*")))
    fake_paths = sorted(glob.glob(os.path.join(base_path, "fake", "*.*")))
    
    # Use provided counts to slice the lists
    if real_count > 0:
        real_paths = real_paths[:real_count]
    if fake_count > 0:
        fake_paths = fake_paths[:fake_count]

    paths = real_paths + fake_paths
    # Use 0 for REAL, 1 for FAKE
    labels = [0] * len(real_paths) + [1] * len(fake_paths)
    
    if not paths:
        print(f"  Loaded {split_name}: 0 paths.")
        return [], []

    paths, labels = shuffle(paths, labels, random_state=42)
    print(f"  Loaded {split_name}: {len(real_paths)} real, {len(fake_paths)} fake. Total: {len(paths)}")
    return paths, labels

# --- Keras Data Generator ---

class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, data_paths, labels, batch_size, input_shape=(299, 299, 3), shuffle=True):
        self.data_paths = data_paths
        self.labels = labels
        self.batch_size = batch_size
        self.input_shape = input_shape
        self.shuffle = shuffle
        
        # Store preprocessors
        self.preprocessors = {
            "densenet": densenet.preprocess_input,
            "efficientnet": efficientnet.preprocess_input,
            "xception": xception.preprocess_input,
        }
        
        self.indices = np.arange(len(self.data_paths))
        self.on_epoch_end()

    def __len__(self):
        # Number of batches per epoch
        return int(np.floor(len(self.data_paths) / self.batch_size))

    def __getitem__(self, index):
        # Get one batch of indices
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        
        # Get paths and labels for this batch
        batch_paths = [self.data_paths[i] for i in batch_indices]
        batch_labels = [self.labels[i] for i in batch_indices]
        
        # Generate the data
        X, y = self.__data_generation(batch_paths, batch_labels)
        return X, y

    def on_epoch_end(self):
        # Shuffle indices after each epoch
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __data_generation(self, batch_paths, batch_labels):
        """Generates data containing batch_size samples"""
        # Load images
        batch_images = []
        for img_path in batch_paths:
            try:
                img = tf_image.load_img(img_path, target_size=self.input_shape[:2])
                img = tf_image.img_to_array(img)
                batch_images.append(img)
            except Exception as e:
                print(f"Warning: Skipping {img_path} due to error: {e}")
                # Add a placeholder
                batch_images.append(np.zeros(self.input_shape))

        batch_images_np = np.array(batch_images, dtype=np.float32)

        # Create the three preprocessed inputs
        X_densenet = self.preprocessors["densenet"](batch_images_np.copy())
        X_efficientnet = self.preprocessors["efficientnet"](batch_images_np.copy())
        X_xception = self.preprocessors["xception"](batch_images_np.copy())
        
        # Labels to categorical
        y = tf.keras.utils.to_categorical(batch_labels, num_classes=2)
        
        # Return a TUPLE for inputs, not a list
        return (X_densenet, X_efficientnet, X_xception), y

print("utils.py written (v4 - Generator Fix).")

Writing utils.py


**TRAINING SCRIPT**

In [5]:
%%writefile train.py
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import gc

# We must import the custom layers for the model to load
import model
import utils

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

class train:
    def __init__(self, train_path, val_path, train_counts, val_counts):
        self.train_path = train_path
        self.val_path = val_path
        self.train_counts = train_counts
        self.val_counts = val_counts
        self.model_save_dir = "models"
        os.makedirs(self.model_save_dir, exist_ok=True)

    def run(self, strategy, epochs, batch_size, steps_per_epoch, val_steps):
        print("--- Starting Training Pipeline (Keras model.fit) ---")
        
        # 1. Load Data Paths
        train_data_paths, train_labels = utils.prepare_dataset_paths(
            os.path.join(self.train_path, "train"), self.train_counts
        )
        val_data_paths, val_labels = utils.prepare_dataset_paths(
            os.path.join(self.val_path, "validation"), self.val_counts
        )
        
        # 2. Create Data Generators
        train_generator = utils.DataGenerator(
            train_data_paths, train_labels, batch_size, shuffle=True
        )
        val_generator = utils.DataGenerator(
            val_data_paths, val_labels, batch_size, shuffle=False
        )
        
        # 3. Build and Compile Model
        print("Building combined model...")
        with strategy.scope():
            combined_model = model.build_combined_model(input_shape=(299, 299, 3))
            
            # --- THIS IS THE FIX: Increased learning rate ---
            combined_model.compile(
                optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), 
                loss='categorical_crossentropy',
                metrics=['accuracy']
            )
        
        print(combined_model.summary())

        # 4. Define Callbacks
        checkpoint_path = os.path.join(self.model_save_dir, "best_combined_model.keras")
        callbacks = [
            ModelCheckpoint(
                filepath=checkpoint_path,
                monitor='val_accuracy',
                save_best_only=True,
                verbose=1
            ),
            EarlyStopping(
                monitor='val_accuracy',
                patience=10, # Stop after 10 epochs of no improvement
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.2,
                patience=3, # Reduce LR after 3 epochs of no improvement
                min_lr=1e-7,
                verbose=1
            )
        ]

        # 5. Train the Model
        print("--- Starting Model Training ---")
        history = combined_model.fit(
            train_generator,
            epochs=epochs,
            steps_per_epoch=steps_per_epoch,
            validation_data=val_generator,
            validation_steps=val_steps,
            callbacks=callbacks
        )
        
        print("--- Training Pipeline Complete ---")
        return history

print("train.py written (v5 - Increased LR to 1e-4).")

Writing train.py


**MAIN ENTRY POINT**
### Cell 13: Code (Write `main.py`)
*This is simplified to pass the correct arguments to the new `train.py`.*

In [6]:
%%writefile main.py
import argparse
from train import train
import json
import os
import tensorflow as tf

def get_distribution_strategy():
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
        strategy = tf.distribute.TPUStrategy(tpu)
    except (ValueError, tf.errors.NotFoundError):
        gpus = tf.config.list_physical_devices('GPU')
        if len(gpus) > 1:
            strategy = tf.distribute.MirroredStrategy()
        elif len(gpus) == 1:
            strategy = tf.distribute.get_strategy()
        else:
            strategy = tf.distribute.get_strategy()
    return strategy

def main():
    parser = argparse.ArgumentParser(description='Train Combined Deepfake Detector')

    # Paths
    parser.add_argument('--train_path', type=str, required=True, help='Base path to training dataset')
    parser.add_argument('--val_path', type=str, required=True, help='Base path to validation dataset')

    # Data counts
    # Updated defaults based on your dataset
    parser.add_argument('--train_real', type=int, default=5605, help='Num real training images')
    parser.add_argument('--train_fake', type=int, default=6028, help='Num fake training images')
    parser.add_argument('--val_real', type=int, default=1200, help='Num real validation images')
    parser.add_argument('--val_fake', type=int, default=1200, help='Num fake validation images')

    # Training params
    parser.add_argument('--epochs', type=int, default=50, help='Number of epochs to train')
    parser.add_argument('--batch_size', type=int, default=32, help='Batch size for training')
    
    args = parser.parse_args()

    train_counts = {'real': args.train_real, 'fake': args.train_fake}
    val_counts = {'real': args.val_real, 'fake': args.val_fake}
    
    # Calculate steps
    total_train = args.train_real + args.train_fake
    total_val = args.val_real + args.val_fake
    steps_per_epoch = total_train // args.batch_size
    val_steps = total_val // args.batch_size


    print("--- Configuration ---")
    print(f"Train Path: {args.train_path}")
    print(f"Val Path: {args.val_path}")
    print(f"Train Counts: {train_counts}")
    print(f"Val Counts: {val_counts}")
    print(f"Epochs: {args.epochs}, Batch Size: {args.batch_size}")
    print(f"Steps per Epoch: {steps_per_epoch}, Validation Steps: {val_steps}")
    print("---------------------\n")
    
    strategy = get_distribution_strategy()
    
    trainer = train(args.train_path, args.val_path, train_counts, val_counts)
    trainer.run(strategy, args.epochs, args.batch_size, steps_per_epoch, val_steps)

if __name__ == '__main__':
    main()
    
print("main.py written (v3 - Keras).")

Writing main.py


**PREDICTION SCRIPT**
### Cell 15: Code (Write `predict.py`)
*This is updated to load the `.keras` model and use the new `DataGenerator`.*


In [7]:
%%writefile predict.py
import argparse
import joblib
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import gc

# We must import the custom layers for the model to load
import model
import utils

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

def main():
    parser = argparse.ArgumentParser(description='Evaluate Combined Deepfake Detector')
    
    # Paths
    parser.add_argument('--test_path', type=str, required=True, help='Base path to test dataset')
    parser.add_argument('--model_dir', type=str, default='models', help='Directory containing saved models')

    # Data counts
    # Updated defaults based on your dataset
    parser.add_argument('--test_real', type=int, default=1200, help='Num real test images')
    parser.add_argument('--test_fake', type=int, default=1200, help='Num fake test images')
    parser.add_argument('--batch_size', type=int, default=32, help='Batch size for feature extraction')

    args = parser.parse_args()

    print("--- Starting Prediction Pipeline ---")
    model_path = os.path.join(args.model_dir, 'best_combined_model.keras')

    # 1. Load saved models
    print(f"Loading trained Keras model from {model_path}...")
    try:
        # Custom objects needed to load the Keras models
        custom_objects = {
            "ModifiedBranch": model.ModifiedBranch, 
            "MainBranch": model.MainBranch, 
            "Attention": model.Attention
        }
        trained_model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)
    except Exception as e:
        print(f"Error: Model file not found or failed to load. {e}")
        print("Please run train.py first to generate the models.")
        return

    # 2. Load Test Data Paths
    test_counts = {'real': args.test_real, 'fake': args.test_fake}
    test_data_paths, test_labels = utils.prepare_dataset_paths(
        os.path.join(args.test_path, "test"), test_counts
    )
    if not test_data_paths:
        print(f"No test images found in {os.path.join(args.test_path, 'test')}. Exiting.")
        return
        
    # 3. Create Test Generator
    # Note: batch_size for prediction can be larger if GPU memory allows
    # We set shuffle=False to keep order for confusion matrix
    test_generator = utils.DataGenerator(
        test_data_paths, 
        test_labels, 
        args.batch_size, 
        shuffle=False
    )
    
    test_steps = int(np.ceil(len(test_labels) / args.batch_size))

    # 4. Run Prediction
    print("Running predictions on test set...")
    probabilities = trained_model.predict(
        test_generator,
        steps=test_steps,
        verbose=1
    )
    
    # Get predictions for all samples, handling the last batch
    all_predictions = np.argmax(probabilities, axis=1)
    all_labels = np.array(test_labels[:len(all_predictions)]) # Ensure labels match prediction count
    all_probs_fake = probabilities[:, 1] # Probability of class 1 (Fake)

    # 5. Report Results
    test_accuracy = accuracy_score(all_labels, all_predictions)
    test_auc = roc_auc_score(all_labels, all_probs_fake)
    cm = confusion_matrix(all_labels, all_predictions)

    print("\n--- FINAL TEST RESULTS ---")
    print(f"  Test Accuracy: {test_accuracy:.4f}")
    print(f"  Test AUC: {test_auc:.4f}")
    print("  Confusion Matrix:")
    print(cm)
    print("-------------------------")

    # Plot and save confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real (0)', 'Fake (1)'], yticklabels=['Real (0)', 'Fake (1)'])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Test Set Confusion Matrix')
    plt.savefig(os.path.join(args.model_dir, 'confusion_matrix.png'))
    print(f"Confusion matrix saved to {args.model_dir}/confusion_matrix.png")

if __name__ == '__main__':
    main()

print("predict.py written (v3 - Keras).")

Writing predict.py


## Part 2: Train the Model

This cell executes the `main.py` script to start the training process. This will now train the Keras model end-to-end, including the attention layers, using `model.fit()`.

**This is a very long-running cell.** It will:
1.  Load all 11,633 training images and 2,400 validation images.
2.  Train the combined model for **50 epochs** (or as specified).
3.  The GPU will be used heavily for `model.fit()`.
4.  It will save the *best* Keras model (`.keras` file) to the `models/` directory based on `val_accuracy`.
### Cell 17: Code (Run Training)

In [8]:
# Set the data path variable
DATA_PATH = "/kaggle/input/1000-videos-split/1000_videos"

print("--- [Step 1] Running Training ---")

# Run main.py. It will use the default counts you provided.
# We pass the paths as arguments.
# We also set a smaller batch size to help with memory on the GPU.
!python main.py \
    --train_path "$DATA_PATH" \
    --val_path "$DATA_PATH" \
    --batch_size 16 \
    --epochs 10

--- [Step 1] Running Training ---
2025-11-12 17:38:45.505389: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762969125.525793     112 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762969125.532234     112 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
model.py written (v7 - Syntax Fix & Fine-Tuning).
utils.py written (v4 - Generator Fix).
train.py written (v5 - Increased LR to 1e-4).
--- Configuration ---
Train Path: /kaggle/input/1000-videos-split/1000_videos
Val Path: /kaggle/input/1000-videos-split/1000_videos
Train Counts: {'real': 5605, 'fake': 6028}
Val Counts: {'real': 1200, 'fake': 1200}
Epochs: 10, Batch Size: 16
Steps per Epoch: 727, Val

## Part 3: Run Prediction

After training is complete, this cell runs `predict.py`. It will load the saved Keras model (`best_combined_model.keras`) from the `models/` folder and evaluate it on your test set.
### Cell 19: Code (Run Prediction)

In [9]:
# Set the data path variable
DATA_PATH = "/kaggle/input/1000-videos-split/1000_videos"

print("--- [Step 2] Running Prediction ---")

# Run predict.py. It will use the default test counts (1200 real, 1200 fake).
!python predict.py \
    --test_path "$DATA_PATH" \
    --batch_size 16

--- [Step 2] Running Prediction ---
2025-11-12 18:05:57.473045: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762970757.494728     405 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762970757.501769     405 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
model.py written (v7 - Syntax Fix & Fine-Tuning).
utils.py written (v4 - Generator Fix).
--- Starting Prediction Pipeline ---
Loading trained Keras model from models/best_combined_model.keras...
I0000 00:00:1762970763.523842     405 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.